# Rede neural com PyTorch

## Classificação binária para diagnóstico de câncer de mama

Base de dados: Breast Cancer Wisconsin (Diagnostic)
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

Esta versão usa PyTorch por meio do `skorch`, que permite treinar a rede com a interface do scikit-learn.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import sklearn
import skorch
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from skorch import NeuralNetBinaryClassifier
from torchinfo import summary as torch_summary


In [2]:
pd.__version__, torch.__version__, sklearn.__version__, skorch.__version__


('3.0.6', '2.14.1+cpu', '1.8.0', '1.4.0')

### Carregar os dados

**Carregue a base de dados, faça a divisão de treino e teste (para isso, utilize a função train_test_split do sklearn), como o tamanho da base de teste de 0.25.**

In [3]:
X = pd.read_csv('dados/dados_breast.csv')
y = pd.read_csv('dados/rotulos_breast.csv')

X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [4]:
X_treinamento.shape, y_treinamento.shape

((426, 30), (426, 1))

In [5]:
X_teste.shape, y_teste.shape

((143, 30), (143, 1))

### Estrutura da Rede Neural Artificial e Teste

In [6]:
from skorch import NeuralNetBinaryClassifier
from sklearn.model_selection import cross_val_score
import torch
import torch.nn as nn


**Crie a RNA com as seguintes configurações:**

**Uma camada de entrada com 30 neurônios;**

**Uma camada oculta densa com 16 neurônios. Porque a camada oculta tem essa quantidade de neurônios?**

Foram usados 16 neurônios como um ponto intermediário entre as 30 entradas e a única saída. Esse valor não é uma regra e pode ser comparado com outras configurações.

**Adicione na camada oculta a função de ativação relu e inicialize os pesos com Random uniform initializer;**

**Adicione a camada de saída com a função de ativação sigmoid. Porque foi utilizada esta função para a saída da RNA?**

A sigmoid limita a saída ao intervalo entre 0 e 1. Neste problema binário, esse valor é tratado como a probabilidade da classe positiva.

**Porque utilizamos a classe chamada Sequential para a RNA?**

A rede segue uma sequência simples de camadas, sem ramificações.

In [7]:
class RedeNeuralModule(nn.Module):
    def __init__(self, n_features=30, neurons=16):
        super().__init__()
        self.oculta = nn.Linear(n_features, neurons)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(neurons, 1)
        nn.init.uniform_(self.oculta.weight)
        nn.init.uniform_(self.saida.weight)

    def forward(self, X):
        x = self.relu(self.oculta(X))
        x = self.saida(x)  # logits (sem sigmoid); NeuralNetBinaryClassifier aplica BCEWithLogitsLoss
        return x.squeeze(-1)

rede_neural = NeuralNetBinaryClassifier(
    module=RedeNeuralModule,
    max_epochs=100,
    batch_size=10,
    verbose=0,
)


**A partir da RNA gerada, explique o que são os valores apresentados na tabela da rede_neural.summary()**

- **Layer (type)**: Nome e tipo de cada camada da rede.
- **Output Shape**: Dimensão da saída de cada camada. `(None, 16)` significa que para cada amostra (None = qualquer quantidade de amostras) há 16 saídas.
- **Param #**: Número de parâmetros treináveis. Para a camada densa de 16 neurônios: (30 entradas × 16 neurônios) + 16 biases = 496 parâmetros. Para a camada de saída: (16 × 1) + 1 bias = 17 parâmetros. Total: 513 parâmetros.
- **Trainable params**: parâmetros que serão atualizados durante o treinamento.
- **Non-trainable params**: parâmetros fixos (ex.: em camadas congeladas).

In [8]:
rede_neural.initialize()
torch_summary(rede_neural.module_, input_size=(1, 30))


Layer (type:depth-idx)                   Output Shape              Param #
RedeNeuralModule                         [1]                       --
├─Linear: 1-1                            [1, 16]                   496
├─ReLU: 1-2                              [1, 16]                   --
├─Linear: 1-3                            [1, 1]                    17
Total params: 513
Trainable params: 513
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

**Adicione um otimizador Adam e especifique a classe loss binário - binary crossentropy e a métrica de acurácia binária.**

**Para que servem os otimizadores? Como o otimizador Adam funciona?**

Os otimizadores servem para atualizar os pesos da rede neural de forma a minimizar a função de perda (loss). O Adam (Adaptive Moment Estimation) combina duas técnicas:
- **Momentum**: acumula gradientes passados para suavizar a atualização (evita oscilações);
- **RMSProp**: adapta a taxa de aprendizado para cada parâmetro individualmente com base na magnitude dos gradientes recentes.
Na prática, isso permite ajustar o passo de cada parâmetro durante o treinamento.

**Depois de estruturados os parâmetros da RNA, utilize a função .fit para fazer o treinamento da rede. Com batch_size=10, quantos batches serão utilizados ao total?**

O conjunto de treinamento tem 426 amostras (75% de 569). Com batch_size=10: ⌈426 / 10⌉ = 43 batches por época × 100 épocas = 4.300 atualizações de pesos no total.

**Defina o número de épocas em que ocorre o treinamento igual a 100.**

In [9]:
_ = rede_neural.set_params(optimizer=torch.optim.Adam, criterion=nn.BCEWithLogitsLoss)


In [10]:
X_tr = X_treinamento.values.astype(np.float32)
y_tr = y_treinamento.values.astype(np.float32).ravel()
_ = rede_neural.fit(X_tr, y_tr)


**Crie uma variável chamada previsoes para realizar a previsão dos dados de teste (X_teste). O resultado da rede deve ser um valor entre 0 e 1. Porque isso acontece?**

A saída fica entre 0 e 1 por causa da sigmoid e é interpretada como a probabilidade da classe positiva.

In [11]:
X_te = X_teste.values.astype(np.float32)
previsoes = rede_neural.predict_proba(X_te)[:, 1]


**Converta o resultado que está em probabilidade para valores binários com um limiar (threshold) de 0.5.**

In [12]:
previsoes = previsoes > 0.5

In [13]:
previsoes

array([False,  True,  True, False,  True,  True,  True,  True,  True,
        True, False,  True,  True, False,  True, False,  True, False,
       False, False, False, False,  True,  True, False,  True,  True,
        True,  True, False,  True, False,  True, False,  True, False,
        True, False,  True, False, False,  True, False,  True, False,
       False,  True,  True,  True, False,  True,  True, False,  True,
        True,  True,  True,  True,  True, False,  True, False,  True,
        True, False,  True, False, False, False,  True, False, False,
        True,  True, False,  True,  True,  True,  True,  True, False,
       False, False,  True, False, False,  True,  True, False, False,
        True, False,  True, False,  True,  True, False,  True,  True,
        True,  True,  True,  True,  True, False,  True, False,  True,
       False,  True,  True, False, False, False,  True,  True, False,
        True,  True,  True,  True,  True, False, False, False, False,
       False,  True,

**Explique o resultado da RNA.**

Com o limiar de 0,5, cada probabilidade é convertida em uma das duas classes. A avaliação mostra a perda e a acurácia no teste.

In [14]:
y_te = y_teste.values.astype(np.float32).ravel()
acuracia = rede_neural.score(X_te, y_te)
print(f'Acurácia binária no teste: {acuracia:.4f}')


Acurácia binária no teste: 0.8811


## Camadas e Otimização da RNA

**Adicione mais uma camada oculta densa com 16 neurônios com a função de ativação relu e inicialize os pesos com Random uniform initializer. Qual o total de parâmetros da RNA agora?**

Com a segunda camada oculta:
- Camada 1: (30×16) + 16 = 496 parâmetros
- Camada 2: (16×16) + 16 = 272 parâmetros
- Saída: (16×1) + 1 = 17 parâmetros
- **Total: 785 parâmetros**

In [15]:
class RedeNeuralModule(nn.Module):
    def __init__(self, n_features=30, neurons=16):
        super().__init__()
        self.oculta1 = nn.Linear(n_features, neurons)
        self.oculta2 = nn.Linear(neurons, neurons)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(neurons, 1)
        for camada in (self.oculta1, self.oculta2, self.saida):
            nn.init.uniform_(camada.weight)

    def forward(self, X):
        x = self.relu(self.oculta1(X))
        x = self.relu(self.oculta2(x))
        x = self.saida(x)
        return x.squeeze(-1)

rede_neural = NeuralNetBinaryClassifier(
    module=RedeNeuralModule,
    max_epochs=100,
    batch_size=10,
    verbose=0,
)
_ = rede_neural.initialize()


In [16]:
torch_summary(rede_neural.module_, input_size=(1, 30))


Layer (type:depth-idx)                   Output Shape              Param #
RedeNeuralModule                         [1]                       --
├─Linear: 1-1                            [1, 16]                   496
├─ReLU: 1-2                              [1, 16]                   --
├─Linear: 1-3                            [1, 16]                   272
├─ReLU: 1-4                              [1, 16]                   --
├─Linear: 1-5                            [1, 1]                    17
Total params: 785
Trainable params: 785
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

**Abaixo são adicionados os parâmetros do otimizador: learning_rate e clipvalue. O que eles fazem?**

- **learning_rate (0.001)**: controla o tamanho do passo dado na descida do gradiente. Valores muito altos fazem o modelo divergir; valores muito baixos tornam o treinamento lento. 0.001 é o valor padrão do Adam.
- **clipvalue (0.5)**: limita o valor máximo dos gradientes durante o backpropagation. Evita o problema de "gradiente explosivo", onde gradientes muito grandes desestabilizam o treinamento.

In [17]:
from skorch.callbacks import Callback

class ClipGradByValue(Callback):
    def on_grad_computed(self, net, named_parameters, **kwargs):
        torch.nn.utils.clip_grad_value_(net.module_.parameters(), clip_value=0.5)

_ = rede_neural.set_params(
    optimizer=torch.optim.Adam,
    optimizer__lr=0.001,
    criterion=nn.BCEWithLogitsLoss,
    callbacks=[ClipGradByValue()],
)


**Teste novamente a RNA. Aumentar a quantidade de camadas melhorou ou piorou os resultados? Explique o que aconteceu com a RNA e porque.**

A segunda camada aumenta a capacidade do modelo, mas não garante melhora no teste. Se o treino melhorar e o teste piorar, a rede provavelmente está sofrendo overfitting.

In [18]:
_ = rede_neural.fit(X_tr, y_tr)


In [19]:
previsoes = rede_neural.predict_proba(X_te)[:, 1]


In [20]:
acuracia = rede_neural.score(X_te, y_te)
print(f'Acurácia binária no teste: {acuracia:.4f}')


Acurácia binária no teste: 0.6434


## K-Fold Cross Validation

**Explique como esta rede foi configurada e como é possível chegar no resultado dela. O que é necessário fazer?**

Na validação cruzada, cada fold assume uma vez o papel de validação e os demais são usados no treino. Com 10 folds, a rede é treinada 10 vezes. O `NeuralNetBinaryClassifier` permite usar o modelo com a API do scikit-learn.

**Calcule também o Desvio Padrão dos resultados para avaliar o modelo. O que é possível concluir com esse resultado?**

O desvio padrão mostra quanto a acurácia mudou entre os folds. Valores altos indicam maior sensibilidade à divisão dos dados.

In [21]:
from skorch import NeuralNetBinaryClassifier
from skorch.callbacks import Callback
from sklearn.model_selection import cross_val_score
import torch.nn as nn
import torch


In [22]:
class ClipGradByValue(Callback):
    def on_grad_computed(self, net, named_parameters, **kwargs):
        torch.nn.utils.clip_grad_value_(net.module_.parameters(), clip_value=0.5)

def criar_rede():
    return NeuralNetBinaryClassifier(
        module=RedeNeuralModule,
        max_epochs=100,
        batch_size=10,
        optimizer=torch.optim.Adam,
        optimizer__lr=0.001,
        criterion=nn.BCEWithLogitsLoss,
        callbacks=[ClipGradByValue()],
        verbose=0,
    )


In [23]:
rede_neural = criar_rede()


In [24]:
X_np = X.values.astype(np.float32)
y_np = y.values.astype(np.float32).ravel()
resultados = cross_val_score(estimator=rede_neural, X=X_np, y=y_np, cv=10, scoring='accuracy')


In [25]:
resultados

array([0.71929825, 0.70175439, 0.59649123, 0.84210526, 0.78947368,
       0.70175439, 0.68421053, 0.68421053, 0.68421053, 0.73214286])

In [26]:
print(f'Média de acurácia: {resultados.mean():.4f}')
print(f'Desvio padrão:     {resultados.std():.4f}')

Média de acurácia: 0.7136
Desvio padrão:     0.0626


## Overfitting e Dropout

**Como vimos, o modelo anterior sofreu overfitting. Para reduzir esse problema, aplicamos a técnica de regularização conhecida como dropout. Aplique o dropout de 20% na primeira e segunda camada oculta. O que acontece com os resultados? E o Desvio Padrão?**

O dropout desativa 20% das ativações durante o treino. A comparação deve ser feita pela média e pelo desvio padrão da validação, sem pressupor que o resultado necessariamente vai melhorar.

In [27]:
class RedeNeuralDropoutModule(nn.Module):
    def __init__(self, n_features=30, neurons=16, dropout=0.2):
        super().__init__()
        self.oculta1 = nn.Linear(n_features, neurons)
        self.drop1 = nn.Dropout(dropout)
        self.oculta2 = nn.Linear(neurons, neurons)
        self.drop2 = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(neurons, 1)
        for camada in (self.oculta1, self.oculta2, self.saida):
            nn.init.uniform_(camada.weight)

    def forward(self, X):
        x = self.drop1(self.relu(self.oculta1(X)))
        x = self.drop2(self.relu(self.oculta2(x)))
        x = self.saida(x)
        return x.squeeze(-1)

def criar_rede_dropout():
    return NeuralNetBinaryClassifier(
        module=RedeNeuralDropoutModule,
        max_epochs=100,
        batch_size=10,
        optimizer=torch.optim.Adam,
        optimizer__lr=0.001,
        criterion=nn.BCEWithLogitsLoss,
        callbacks=[ClipGradByValue()],
        verbose=0,
    )


In [28]:
rede_neural = criar_rede_dropout()


In [29]:
resultados = cross_val_score(estimator=rede_neural, X=X_np, y=y_np, cv=10, scoring='accuracy')


In [30]:
resultados

array([0.73684211, 0.71929825, 0.61403509, 0.71929825, 0.77192982,
       0.85964912, 0.78947368, 0.75438596, 0.68421053, 0.85714286])

In [31]:
print(f'Média de acurácia (com Dropout): {resultados.mean():.4f}')
print(f'Desvio padrão (com Dropout):     {resultados.std():.4f}')

Média de acurácia (com Dropout): 0.7506
Desvio padrão (com Dropout):     0.0710


## Tuning dos Hiperparâmetros (técnicas de ajuste)

O ajuste de hiperparâmetros compara configurações definidas antes do treinamento.
Exemplos de hiperparâmetros:

- Taxa de aprendizado (learning rate)
- Número de árvores em um RandomForest
- Número de camadas ou neurônios em uma rede neural
- Tipo de kernel em um SVM

Algumas formas de fazer essa busca são:

- Grid Search
- Random Search
- Bayesian Optimization, etc.

**Descreva como a RNA foi configurada para fazer o processo de tuning.**

A classe da rede recebe os parâmetros que serão comparados. O `NeuralNetBinaryClassifier` adapta o modelo à interface do scikit-learn, e o `GridSearchCV` avalia as combinações com cinco folds.

In [32]:
class RedeNeuralTunavelModule(nn.Module):
    def __init__(self, n_features=30, neurons=16, activation=nn.ReLU):
        super().__init__()
        self.oculta1 = nn.Linear(n_features, neurons)
        self.drop1 = nn.Dropout(0.2)
        self.oculta2 = nn.Linear(neurons, neurons)
        self.drop2 = nn.Dropout(0.2)
        self.ativacao = activation()
        self.saida = nn.Linear(neurons, 1)

    def forward(self, X):
        x = self.drop1(self.ativacao(self.oculta1(X)))
        x = self.drop2(self.ativacao(self.oculta2(x)))
        x = self.saida(x)
        return x.squeeze(-1)

rede_neural = NeuralNetBinaryClassifier(
    module=RedeNeuralTunavelModule,
    criterion=nn.BCEWithLogitsLoss,
    verbose=0,
)


In [33]:
# rede_neural já criado na célula acima


In [34]:
parametros = {
    'batch_size': [10, 30],
    'max_epochs': [50, 100],
    'optimizer': [torch.optim.Adam, torch.optim.SGD],
    'module__activation': [nn.ReLU, nn.Tanh],
    'module__neurons': [16, 8],
}


In [35]:
# caso tiver problema com a configuração anterior, testar essa (espaço de busca reduzido)
parametros = {
    'batch_size': [10, 30],
    'max_epochs': [50],
    'optimizer': [torch.optim.Adam],
    'module__activation': [nn.ReLU],
    'module__neurons': [16],
}


In [36]:
parametros

{'batch_size': [10, 30],
 'max_epochs': [50],
 'optimizer': [torch.optim.adam.Adam],
 'module__activation': [torch.nn.modules.activation.ReLU],
 'module__neurons': [16]}

In [37]:
grid_search = GridSearchCV(estimator=rede_neural, param_grid=parametros,
                           scoring='accuracy', cv=5)


In [38]:
grid_search = grid_search.fit(X_np, y_np)


In [39]:
print(grid_search)

GridSearchCV(cv=5,
             estimator=NeuralNetBinaryClassifier(_params_to_validate=set(), batch_size=128, callbacks=None, compile=False, dataset=<class 'skorch.dataset.Dataset'>, device='cpu', iterator_train=<class 'torch.utils.data.dataloader.DataLoader'>, iterator_valid=<class 'torch.utils.data.dataloader.DataLoader'>, lr=0.01, max_epochs=10, module=<class '__main__.Re...Module'>, optimizer=<class 'torch.optim.sgd.SGD'>, predict_nonlinearity='auto', torch_load_kwargs=None, use_caching='auto', verbose=0, warm_start=False),
             param_grid={'batch_size': [10, 30], 'max_epochs': [50],
                         'module__activation': [<class 'torch.nn.modules.activation.ReLU'>],
                         'module__neurons': [16],
                         'optimizer': [<class 'torch.optim.adam.Adam'>]},
             scoring='accuracy')


In [40]:
melhores_parametros = grid_search.best_params_
print(melhores_parametros)

{'batch_size': 30, 'max_epochs': 50, 'module__activation': <class 'torch.nn.modules.activation.ReLU'>, 'module__neurons': 16, 'optimizer': <class 'torch.optim.adam.Adam'>}


In [41]:
melhor_precisao = grid_search.best_score_
print(melhor_precisao)

0.7682657972364539


**É possível melhorar ainda mais a acurácia da RNA? Como?**

Sim. Algumas estratégias possíveis:
- **Aumentar o espaço de busca do Grid Search**: testar mais combinações de neurônios, camadas, otimizadores e taxas de aprendizado.
- **Normalização dos dados**: aplicar StandardScaler ou MinMaxScaler nas features pode acelerar a convergência e melhorar a acurácia.
- **Aumentar épocas**: mais épocas permitem que a rede aprenda melhor, desde que combinado com dropout para evitar overfitting.
- **Adicionar mais camadas**: arquiteturas mais profundas capturam padrões mais complexos.
- **Batch Normalization**: normaliza as ativações entre camadas, estabilizando e acelerando o treinamento.
- **Early Stopping**: parar o treinamento quando a loss de validação para de melhorar, evitando overfitting.

# Para salvar o seu modelo de RNA

In [42]:
# Treina o modelo final com os melhores parâmetros antes de salvar
classificador = NeuralNetBinaryClassifier(
    module=RedeNeuralDropoutModule,
    max_epochs=100,
    batch_size=10,
    optimizer=torch.optim.Adam,
    optimizer__lr=0.001,
    criterion=nn.BCEWithLogitsLoss,
    callbacks=[ClipGradByValue()],
    verbose=0,
)
_ = classificador.fit(X_tr, y_tr)


In [43]:
import os

# Salva na pasta do usuário (evita problemas de permissão em pastas do sistema/OneDrive)
caminho_modelo = os.path.join(os.path.expanduser('~'), 'classificador_breast.pt')
print('Salvando em:', caminho_modelo)
classificador.save_params(f_params=caminho_modelo)


Salvando em: C:\Users\AMD\classificador_breast.pt


In [44]:
classificador_novo = NeuralNetBinaryClassifier(
    module=RedeNeuralDropoutModule,
    criterion=nn.BCEWithLogitsLoss,
)
classificador_novo.initialize()
classificador_novo.load_params(f_params=caminho_modelo)


In [45]:
torch_summary(classificador_novo.module_, input_size=(1, 30))


Layer (type:depth-idx)                   Output Shape              Param #
RedeNeuralDropoutModule                  [1]                       --
├─Linear: 1-1                            [1, 16]                   496
├─ReLU: 1-2                              [1, 16]                   --
├─Dropout: 1-3                           [1, 16]                   --
├─Linear: 1-4                            [1, 16]                   272
├─ReLU: 1-5                              [1, 16]                   --
├─Dropout: 1-6                           [1, 16]                   --
├─Linear: 1-7                            [1, 1]                    17
Total params: 785
Trainable params: 785
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00